# COMP4153 Natural Language Processing
# Lab 5 — Information Extraction and Structured NLP

**Version:** Student  
**Duration:** approximately 2 hours  
**Environment:** local Jupyter Notebook or Google Colab  
**Device:** automatically uses a CUDA GPU when available; otherwise CPU  
**Internet:** not required for the core lab  
**Lecture basis:** **Lecture 5 — Information Extraction and Structured NLP**

This notebook is aligned to the supplied Lecture 5 and follows its conceptual progression:

```text
raw text
   ↓
entity extraction
   ├─ BIO / token tagging
   └─ span-based extraction
   ↓
entity linking
   ↓
relation extraction
   ├─ ordered entity pairs
   ├─ entity markers
   └─ pipeline vs joint extraction
   ↓
event extraction
   ├─ trigger detection
   └─ role-filled arguments
   ↓
schema-based record extraction
   ├─ schema
   ├─ nullable fields
   ├─ repeated records
   └─ parse + validate + store/retry
   ↓
evaluation and error analysis
```

By the end of the lab, you should be able to:

- formulate entity extraction as **BIO sequence tagging** or **span classification**;
- explain the difference between **recognition, typing and linking**;
- distinguish traditional NER from broader **domain-specific entity extraction**;
- build relation-classification inputs using **HEAD/TAIL entity markers**;
- explain why relation **context and direction** matter;
- explain **pipeline error propagation** and the motivation for joint extraction;
- represent events using **triggers and semantic-role arguments**;
- map event information into an application **record schema**;
- design schemas with **nullable fields and repeated records** when needed;
- treat LLM-produced JSON as a **candidate record that still requires validation**;
- evaluate entities, relations, events and records using an output-unit-appropriate correctness rule;
- diagnose boundary, type, linking, relation, direction, missing-argument, unsupported-generation and schema errors.

> The small PyTorch models are teaching models. They are deliberately small enough to run quickly on CPU while exposing the same structures discussed in Lecture 5.


## 0. Lab conventions and task schema

The core notebook should take only a few minutes of actual computation. Most of the lab time should be spent **reading, coding and interpreting predictions**.

We use **word-level tokens** to keep BIO boundaries easy to inspect. Real Transformer systems usually use **subword tokenisation**, which requires aligning word-level entity labels to subword pieces.

Our classroom entity ontology is:

| Type | Meaning |
|---|---|
| `ORG` | organisation or team |
| `PER` | person |
| `PROJ` | project, system or artefact |
| `EVENT` | showcase, expo or named event |
| `DATE` | date expression |

A BIO tag combines **boundary information** with **entity type**:

```text
B-ORG  = beginning of an organisation mention
I-ORG  = inside the same organisation mention
O      = outside any target entity
```

Throughout the lab, exact entity spans use **0-based inclusive token indices**.


In [ ]:
# Cell A0.1 — Imports, reproducibility and CPU/GPU selection

# READING GUIDE: `import` loads reusable Python tools; we do not write them ourselves.
# A variable stores a value for later (for example, SEED stores the number 7).
# Run this setup cell first: later cells need the libraries and `device` defined here.


import json
import html
import os
import random
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display, HTML

# Using the same seed makes random training behaviour more reproducible.
SEED = 7

def set_seed(seed=SEED):
    """Reset the random generators used by this notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # This line is harmless on a CPU-only machine.
    # The CUDA check is True only when a supported NVIDIA GPU is available.
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

# Keep one `device` variable and use it everywhere.
# This makes the same notebook work on CPU and CUDA without code changes.
# The expression "A if condition else B" chooses GPU when possible, CPU otherwise.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print("Device: CUDA GPU")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Device: CPU")

    # Tiny teaching models can become slower if PyTorch creates many CPU threads.
    # Limiting the thread count makes classroom runtime more predictable.
    available_threads = os.cpu_count() or 1
    torch.set_num_threads(min(4, available_threads))
    print("PyTorch CPU threads:", torch.get_num_threads())

print("PyTorch version:", torch.__version__)


# Part A — Guided walkthrough

We will build the pipeline one stage at a time:

$$
\text{text}
\rightarrow
\text{entities}
\rightarrow
\text{linked identities}
\rightarrow
\text{relations}
\rightarrow
\text{event/record}
\rightarrow
\text{validation}.
$$

At every stage, ask three questions:

1. **What is the output unit?** A token label, span, tuple, event argument, or field-value pair?
2. **What counts as correct?**
3. **What failure modes can occur?**

This is important because an IE system can have a reasonable overall score while still making errors that are unacceptable for the downstream application.


### A1. Entity extraction as BIO sequence tagging

Consider the announcement:

> **Nova Robotics** will demonstrate a **solar-powered rover** at **Innovation Showcase** on **18 May**.

An entity extractor must solve two linked problems:

1. **Boundary detection:** which tokens belong to the mention?
2. **Type classification:** what type is the mention?

BIO combines both decisions in one label sequence.

In [ ]:
# Cell A1.1 — Lecture 5 running example and BIO encoding

# GOAL: match each word (token) with its correct entity label.
# Python lists use square brackets [...]; the first token has index 0.
# BIO: B = beginning, I = inside, O = outside a named entity.


# Lecture 5 running example (first sentence).
walk_tokens = [
    "Nova", "Robotics", ",", "a", "student", "team", "from", "UNNC", ",",
    "will", "demonstrate", "a", "solar-powered", "rover", "at", "the",
    "Ningbo", "Campus", "Innovation", "Showcase", "on", "18", "May", "."
]

# For the BIO walkthrough we use non-overlapping entity spans.
# The named showcase is treated as one EVENT mention.
# IMPORTANT: this list must be the same length as walk_tokens.
# "Nova Robotics" gets B-ORG, I-ORG because it is one two-word organisation.
walk_labels = [
    "B-ORG", "I-ORG", "O", "O", "O", "O", "O", "B-ORG", "O",
    "O", "O", "O", "B-PROJ", "I-PROJ", "O", "O",
    "B-EVENT", "I-EVENT", "I-EVENT", "I-EVENT", "O",
    "B-DATE", "I-DATE", "O"
]

print(f"{'TOKEN':22s} {'BIO LABEL':12s}")
print("-" * 36)

# `zip` visits corresponding entries from both lists together.
# e.g. first pass: token="Nova", label="B-ORG".
for token, label in zip(walk_tokens, walk_labels):
    print(f"{token:22s} {label:12s}")


#### Decode BIO tags back into spans

For official entity-level evaluation, a prediction is normally correct only when **both the boundary and the type match exactly**.

We use **0-based inclusive token indices** in the code. For example:

```text
("Nova Robotics", "ORG", 0, 1)
```

In [ ]:
# Cell A1.2 — BIO decoder

# GOAL: group token-by-token BIO labels into whole entity mentions.
# `def` creates a reusable function; `return` gives its result back to the caller.
# Example output: ("Nova Robotics", "ORG", 0, 1), with inclusive indices.


def decode_bio(tokens, labels):
    # Convert BIO labels into:
    # (surface_text, entity_type, start_index, end_index)
    #
    # Robustness choice: an orphan I-X is treated as a new entity start
    # so malformed model outputs can still be inspected.
    # We build a list of complete entity spans, not individual token labels.
    entities = []
    i = 0

    # `while` repeats until we have looked at every token.
    # `i` is the index of the next token we have not processed.
    while i < len(tokens):
        label = labels[i]

        # O means this token is outside every target entity.
        if label == "O":
            i += 1
            continue  # Skip this token and return to the top of the loop.

        if "-" not in label:
            i += 1
            continue

        # Example: "B-ORG" -> prefix="B", entity_type="ORG".
        prefix, entity_type = label.split("-", 1)

        if prefix not in {"B", "I"}:
            i += 1
            continue

        start = i
        i += 1

        # Continue consuming I-TYPE tags only while the type stays the same.
        # This is what reconstructs a multi-token mention such as "Nova Robotics".
        while i < len(tokens) and labels[i] == f"I-{entity_type}":
            i += 1

        end = i - 1
        # Slicing [start:end + 1] includes the final word of the mention.
        # `.join` connects its words into one readable string.
        text = " ".join(tokens[start:end + 1])
        # Store the surface string, type, and exact inclusive token boundary.
        entities.append((text, entity_type, start, end))

    return entities


walk_entities = decode_bio(walk_tokens, walk_labels)
walk_entities

In [ ]:
# Cell A1.3 — Small visualisation helper

# GOAL: make the extracted entities easier to see in the notebook output.
# This cell only displays results; it does NOT train or change the NLP model.
# Colours are stored in a dictionary: entity type -> background colour.


TYPE_STYLE = {
    "ORG": "#d8f3dc",
    "PER": "#dbeafe",
    "PROJ": "#fef3c7",
    "EVENT": "#ede9fe",
    "DATE": "#cffafe",
}

def render_entities(tokens, entities):
    # Render token spans with entity labels.
    # Index each entity by its start position so we can render a complete
    # multi-token span once rather than colouring each token independently.
    # This dictionary comprehension makes it fast to look up a span by its start.
    start_map = {start: (text, typ, start, end) for text, typ, start, end in entities}
    pieces = []
    i = 0

    while i < len(tokens):
        if i in start_map:
            text, typ, start, end = start_map[i]
            # `.get(key, default)` returns grey when the type has no preset colour.
            bg = TYPE_STYLE.get(typ, "#eeeeee")
            pieces.append(
                f'<span style="background:{bg}; padding:4px 6px; margin:2px; '
                f'border-radius:6px; display:inline-block;">'
                f'{html.escape(text)} '
                f'<b style="font-size:0.72em;">{html.escape(typ)}</b></span>'
            )
            # Skip all words in this entity: it has already been displayed as one span.
            i = end + 1
        else:
            pieces.append(html.escape(tokens[i]))
            i += 1

    # `html.escape` above ensures text is displayed as text, not interpreted as HTML.
    display(HTML(" ".join(pieces)))

render_entities(walk_tokens, walk_entities)

### A1.1 Build a small BiLSTM BIO tagger

As in Lecture 5, the model predicts a token-level distribution

$$p(y_i\mid x)=\operatorname{softmax}(Wh_i+b),$$

where $h_i=[\overrightarrow{h}_i;\overleftarrow{h}_i]$ is the **bidirectional LSTM context representation** introduced by the sequence models in Lecture 4.

The model consists of (1) learned word embeddings, (2) a **BiLSTM** that reads the sentence in both directions, and (3) a linear BIO classifier shared across positions. Padded positions are ignored by the loss. This is a small, scratch-trained teaching example, **not** a reliable production extractor.

In [ ]:
# Cell A1.4 — Tiny teaching dataset

# GOAL: prepare a few labelled examples from which the tagger can learn.
# Each example is a pair: (list of words, list of correct BIO labels).
# The lists must line up: tokens[i] and labels[i] describe the same word.
# These examples are deliberately tiny, so the model is a teaching tool only.


# Small controlled training set.
# Each example contains:
#   tokens -> the input sequence
#   BIO labels -> one gold label for every token
token_examples = [
    (
        ["Nova", "Robotics", "will", "demonstrate", "a", "solar-powered", "rover",
         "at", "the", "Ningbo", "Campus", "Innovation", "Showcase", "on", "18", "May"],
        ["B-ORG", "I-ORG", "O", "O", "O", "B-PROJ", "I-PROJ",
         "O", "O", "B-EVENT", "I-EVENT", "I-EVENT", "I-EVENT", "O", "B-DATE", "I-DATE"]
    ),

    (
        ["Nova","Robotics","will","demonstrate","a","solar-powered","rover","at",
         "Innovation","Showcase","on","18","May"],
        ["B-ORG","I-ORG","O","O","O","B-PROJ","I-PROJ","O",
         "B-EVENT","I-EVENT","O","B-DATE","I-DATE"]
    ),
    (
        ["Lin","Mei","leads","Nova","Robotics"],
        ["B-PER","I-PER","O","B-ORG","I-ORG"]
    ),
    (
        ["Aero","Lab","built","a","delivery","drone","for","Tech","Expo"],
        ["B-ORG","I-ORG","O","O","B-PROJ","I-PROJ","O","B-EVENT","I-EVENT"]
    ),
    (
        ["Maya","Chen","presented","the","vision","system","at","Research","Day"],
        ["B-PER","I-PER","O","O","B-PROJ","I-PROJ","O","B-EVENT","I-EVENT"]
    ),
    (
        ["Green","Mobility","Team","will","show","an","electric","kart","at",
         "Innovation","Fair","on","21","June"],
        ["B-ORG","I-ORG","I-ORG","O","O","O","B-PROJ","I-PROJ","O",
         "B-EVENT","I-EVENT","O","B-DATE","I-DATE"]
    ),
    (
        ["Noah","Wang","joined","Quantum","Works"],
        ["B-PER","I-PER","O","B-ORG","I-ORG"]
    ),
    (
        ["Robotics","Club","tested","the","inspection","robot","at","Maker","Festival"],
        ["B-ORG","I-ORG","O","O","B-PROJ","I-PROJ","O","B-EVENT","I-EVENT"]
    ),
    (
        ["Sofia","Li","designed","a","medical","chatbot","for","Student","Showcase"],
        ["B-PER","I-PER","O","O","B-PROJ","I-PROJ","O","B-EVENT","I-EVENT"]
    ),
    (
        ["Ocean","AI","will","launch","Reef","Watch","at","Blue","Tech","Expo","on","3","July"],
        ["B-ORG","I-ORG","O","O","B-PROJ","I-PROJ","O","B-EVENT","I-EVENT","I-EVENT",
         "O","B-DATE","I-DATE"]
    ),
    (
        ["Ethan","Zhou","works","with","Smart","Farming","Lab"],
        ["B-PER","I-PER","O","O","B-ORG","I-ORG","I-ORG"]
    ),
    (
        ["Data","Science","Society","demoed","a","campus","assistant","at","Open","Day"],
        ["B-ORG","I-ORG","I-ORG","O","O","B-PROJ","I-PROJ","O","B-EVENT","I-EVENT"]
    ),
    (
        ["Iris","Sun","introduced","the","path","planner","at","Demo","Day","on","12","April"],
        ["B-PER","I-PER","O","O","B-PROJ","I-PROJ","O","B-EVENT","I-EVENT",
         "O","B-DATE","I-DATE"]
    ),
]

# Fixed label vocabulary.  The neural network predicts one of these
# classes independently at every token position.
LABELS = [
    "O",
    "B-ORG", "I-ORG",
    "B-PER", "I-PER",
    "B-PROJ", "I-PROJ",
    "B-EVENT", "I-EVENT",
    "B-DATE", "I-DATE",
]

# PyTorch losses use integer class IDs rather than strings.
# `enumerate` pairs each label with a number: "O" -> 0, "B-ORG" -> 1, ...
label_to_id = {label: i for i, label in enumerate(LABELS)}
id_to_label = {i: label for label, i in label_to_id.items()}

print("Examples:", len(token_examples))
print("BIO labels:", LABELS)

In [ ]:
# Cell A1.5 — Vocabulary and batching

# GOAL: turn words and labels into numbers that PyTorch can process.
# Neural networks cannot directly use strings such as "Nova" or "B-ORG".
# A "batch" groups sentences together; shorter ones need padding to match lengths.


# <PAD> fills unused positions; <UNK> represents unseen words.
SPECIAL_TOKENS = ["<PAD>", "<UNK>"]
# Build a word-level vocabulary from the tiny training set.
# Unseen words at test time map to <UNK>, which we will later see is a limitation.
# `{...}` here collects unique words; `sorted` gives a consistent order.
vocab = SPECIAL_TOKENS + sorted({
    token
    for sentence, _ in token_examples
    for token in sentence
})

token_to_id = {token: i for i, token in enumerate(vocab)}
PAD_ID = token_to_id["<PAD>"]
UNK_ID = token_to_id["<UNK>"]

# All sentences will be extended to the length of the longest sentence.
MAX_LEN = max(len(sentence) for sentence, _ in token_examples)

def make_token_batch(examples):
    input_ids, label_ids, attention_masks = [], [], []

    for tokens, labels in examples:
        # Convert strings to integer token IDs.
        # get(..., UNK_ID) maps an unseen word to the shared <UNK> representation.
        ids = [token_to_id.get(tok, UNK_ID) for tok in tokens]
        y = [label_to_id[label] for label in labels]

        # `pad_n` is how many extra slots this shorter sentence needs.
        pad_n = MAX_LEN - len(ids)

        input_ids.append(ids + [PAD_ID] * pad_n)
        # PyTorch CrossEntropyLoss(ignore_index=-100) ignores padded positions.
        # We still need padding to make all sequences in the batch the same length.
        label_ids.append(y + [-100] * pad_n)  # ignored by CrossEntropyLoss
        # 1 = real token, 0 = padding.  The BiLSTM uses sequence lengths to exclude padded tokens.
        attention_masks.append([1] * len(ids) + [0] * pad_n)

    # A tensor is PyTorch's numeric array. Here the shape is
    # [number_of_sentences, longest_sentence_length].
    # `device=device` puts all arrays on the same CPU or GPU as the model.
    return (
        torch.tensor(input_ids, dtype=torch.long, device=device),
        torch.tensor(label_ids, dtype=torch.long, device=device),
        torch.tensor(attention_masks, dtype=torch.bool, device=device),
    )

X_tok, Y_tok, M_tok = make_token_batch(token_examples)

print("Vocabulary size:", len(vocab))
print("Batch shape:", X_tok.shape)

In [ ]:
# Cell A1.6 — BiLSTM token classifier

# GOAL: define a small neural network that predicts a BIO label per token.
# A class is a reusable model blueprint; __init__ sets up its layers.
# `forward` describes the steps applied to input data to make predictions.


class TinyTokenClassifier(nn.Module):
    def __init__(self, vocab_size, num_labels, embedding_dim=48, hidden_size=40):
        super().__init__()
        # Embedding: replace each token ID with a short trainable number vector.
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        # BiLSTM: read the sentence left-to-right AND right-to-left.
        # Each direction contributes hidden_size values per token.
        self.encoder = nn.LSTM(embedding_dim, hidden_size, num_layers=1,
                               bidirectional=True, batch_first=True)
        # Linear layer: turn each token's 2*hidden_size values into label scores.
        self.classifier = nn.Linear(2 * hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, return_hidden=False):
        # Shape changes: [batch, tokens] -> [batch, tokens, embedding_dim].
        embeddings = self.token_embedding(input_ids)
        # Packed sequences stop padding from changing the backward LSTM states.
        lengths = attention_mask.long().sum(dim=1).cpu().clamp(min=1)
        packed = nn.utils.rnn.pack_padded_sequence(
            embeddings, lengths, batch_first=True, enforce_sorted=False)
        packed_hidden, _ = self.encoder(packed)
        contextual, _ = nn.utils.rnn.pad_packed_sequence(
            packed_hidden, batch_first=True, total_length=input_ids.size(1))
        # `logits` are raw class scores; the highest score wins at prediction time.
        # Shape: [batch, tokens, num_labels].
        logits = self.classifier(contextual)
        return (logits, contextual) if return_hidden else logits


token_model = TinyTokenClassifier(vocab_size=len(vocab), num_labels=len(LABELS)).to(device)
print(token_model)

#### Training objective

For each non-padding token, we minimise cross-entropy between the predicted distribution and the gold BIO label.

Because the dataset is tiny, the model will **overfit**. That is intentional here: first verify that the architecture can learn the representation. We will then test it on a new announcement and observe why a scratch-built vocabulary generalises poorly.

In [ ]:
# Cell A1.7 — Train the token classifier

# GOAL: teach the token classifier using examples and their correct labels.
# An "epoch" is one complete training pass over our small dataset.
# Lower loss generally means the model's predictions better match training labels.


set_seed()

optimizer = torch.optim.Adam(
    token_model.parameters(),
    lr=0.01,
)

# Padding labels were set to -100 in make_token_batch(), so those positions
# make no contribution to the training loss.
criterion = nn.CrossEntropyLoss(ignore_index=-100)

history = []
token_train_start = time.perf_counter()

# `range(121)` means epochs 0, 1, ..., 120 (121 updates in total).
for epoch in range(121):
    token_model.train()

    # PyTorch accumulates gradients by default, so clear the previous update.
    optimizer.zero_grad()

    # logits shape: [batch, sequence_length, number_of_BIO_labels]
    logits = token_model(X_tok, M_tok)

    # CrossEntropyLoss expects [N, C] predictions and [N] gold labels.
    # Flatten batch and token positions into one dimension.
    loss = criterion(
        logits.reshape(-1, len(LABELS)),
        Y_tok.reshape(-1),
    )

    # Compute gradients, then update all trainable model parameters.
    # Backpropagation finds how each parameter should change to lower the loss.
    loss.backward()
    # Gradient clipping helps avoid excessively large training updates.
    torch.nn.utils.clip_grad_norm_(token_model.parameters(), max_norm=1.0)
    # Adam uses the gradients to adjust the model's trainable weights.
    optimizer.step()

    history.append(float(loss.detach().cpu()))

    # `%` gives the remainder: print every 30 epochs (and at the last one).
    if epoch % 30 == 0 or epoch == 120:
        print(f"epoch={epoch:3d} | loss={loss.item():.4f}")

token_train_seconds = time.perf_counter() - token_train_start

print(f"\nToken-classifier training time: {token_train_seconds:.2f} seconds")


In [ ]:
# Cell A1.8 — Inspect predictions

# GOAL: check what the trained model predicts on a training sentence.
# GOLD is the correct label; PRED is the model's predicted label.
# A tick means the labels agree, and a cross means they disagree.


# `eval()` switches off training-only behaviour for inference.
token_model.eval()

# We only inspect predictions, so gradients are unnecessary and waste memory.
with torch.no_grad():
    # `argmax` selects the label ID with the highest score for each token.
    pred_ids = token_model(X_tok, M_tok).argmax(dim=-1)

example_index = 0
tokens, gold_labels = token_examples[example_index]
pred_labels = [
    id_to_label[int(i)]
    for i in pred_ids[example_index, :len(tokens)].cpu()
]

print(f"{'TOKEN':18s} {'GOLD':10s} {'PRED':10s}")
print("-" * 42)

for tok, gold, pred in zip(tokens, gold_labels, pred_labels):
    marker = "✓" if gold == pred else "✗"
    print(f"{tok:18s} {gold:10s} {pred:10s} {marker}")

pred_entities = decode_bio(tokens, pred_labels)

print("\nDecoded predicted entities:")
for e in pred_entities:
    print(e)

render_entities(tokens, pred_entities)

# Token-level training accuracy is useful only as a sanity check here.
# Exact entity-level evaluation is stricter and is computed in the next section.
with torch.no_grad():
    train_logits = token_model(X_tok, M_tok)
    train_pred = train_logits.argmax(dim=-1)
    # Ignore padded positions; otherwise accuracy would be misleading.
    valid = Y_tok != -100
    train_token_accuracy = (
        (train_pred[valid] == Y_tok[valid]).float().mean().item()
    )

print(f"\nTraining-token accuracy: {train_token_accuracy:.3f}")


### A1.2 Exact-match entity evaluation

For an entity item, **boundary + type** must both match.

We therefore compare sets of tuples such as:

```text
(surface_text, entity_type, start_index, end_index)
```

Then:

$$
P=\frac{|\text{correct predictions}|}{|\text{predictions}|}, \qquad
R=\frac{|\text{correct predictions}|}{|\text{gold items}|},
$$

$$
F_1=2\frac{PR}{P+R}.
$$

In [ ]:
# Cell A1.9 — Precision, recall and F1

# GOAL: score whole extracted entities, not just individual words.
# `set(...)` allows us to compare unique predicted and correct entity tuples.
# An entity only counts as correct when its full span AND its type match.


def prf1(predicted_items, gold_items):
    pred = set(predicted_items)
    gold = set(gold_items)

    # Exact set intersection: a span counts only if surface boundary AND type
    # exactly match a gold item.
    # `&` means set intersection: items that appear in BOTH sets.
    correct = len(pred & gold)

    # Precision: of everything predicted, how much was correct?
    precision = correct / len(pred) if pred else 0.0
    # Recall: of everything that should have been found, how much was found?
    recall = correct / len(gold) if gold else 0.0
    # F1 balances precision and recall; it is 0 when both are 0.
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "correct": correct,
        "predicted": len(pred),
        "gold": len(gold),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


gold_entities = decode_bio(tokens, gold_labels)
entity_scores = prf1(pred_entities, gold_entities)

entity_scores

> **Important:** A high score here only proves that this tiny model can memorise the tiny teaching dataset. It is **not** evidence of a good production NER model.

Now inspect how exact match reacts to boundary and type errors.

In [ ]:
# Cell A1.10 — Deliberately introduce errors

# GOAL: see why nearly correct entity predictions still fail exact matching.
# Here we deliberately shorten one entity and mislabel the other.


gold_demo = [
    ("solar-powered rover", "PROJ", 5, 6),
    ("Innovation Showcase", "EVENT", 8, 9),
]

# Changing either the boundary or the type makes a predicted tuple different.
pred_demo = [
    ("rover", "PROJ", 6, 6),                 # boundary error
    ("Innovation Showcase", "ORG", 8, 9),    # type error
]

print(prf1(pred_demo, gold_demo))
print("\nNeither prediction counts as exactly correct.")

### A2. The span-based viewpoint

BIO predicts **one main label per token**. A span model instead predicts a candidate directly as:

$$
(i,j,t).
$$

This is useful when:

- we want to reason explicitly about span boundaries;
- entities can overlap or nest;
- downstream relation extraction already wants whole spans.

A simple span model first enumerates candidate spans up to a maximum length $L$.

In [ ]:
# Cell A2.1 — Generate candidate spans

# GOAL: list possible contiguous spans (groups of neighbouring tokens).
# Example: for ["Lin", "Mei"], candidates include "Lin", "Mei", "Lin Mei".
# max_span_length limits how many tokens a candidate may contain.


def candidate_spans(tokens, max_span_length=3):
    # Return (start_index, end_index, text) with inclusive indices.
    spans = []

    # Enumerate every possible start position.
    for start in range(len(tokens)):
        # Only allow spans whose length is <= max_span_length.
        # Larger limits increase recall but also increase the number of candidates.
        # `end` is inclusive; min(...) stops before we exceed the limit.
        for end in range(
            start,
            min(len(tokens), start + max_span_length)
        ):
            spans.append(
                (start, end, " ".join(tokens[start:end + 1]))
            )

    # No model is used here: we generate candidates, not predictions.
    return spans


span_sentence = ["Lin", "Mei", "leads", "the", "solar", "rover", "team"]
spans = candidate_spans(span_sentence, max_span_length=3)

print("Number of candidates:", len(spans))
print("First 12 candidates:")
for s in spans[:12]:
    print(s)

A common span representation can combine:

- the contextual representation at the left boundary $h_i$;
- the contextual representation at the right boundary $h_j$;
- a pooled representation over the whole span.

For example:

$$
s_{ij}=[h_i;h_j;\operatorname{mean}(h_i,\ldots,h_j)].
$$

The next cell uses the hidden states from our trained encoder to construct exactly that representation. We do **not** train a second span classifier because the goal here is to understand the representation and candidate-generation trade-off.

In [ ]:
# Cell A2.2 — Construct one span representation from contextual hidden states

# GOAL: represent an entire candidate entity span as one numeric vector.
# A "hidden state" is the numerical representation learned for each token.
# The span vector combines information from its start, end and interior.


def encode_tokens_with_token_model(tokens):
    ids = torch.tensor(
        [[token_to_id.get(tok, UNK_ID) for tok in tokens]],
        dtype=torch.long,
        device=device,
    )
    # A mask of True values says every token in this one sentence is real.
    mask = torch.ones_like(ids, dtype=torch.bool)

    token_model.eval()
    # We only need contextual representations; no parameter update is required.
    with torch.no_grad():
        _, hidden = token_model(ids, mask, return_hidden=True)

    # [0] removes the batch dimension because we encoded only one sentence.
    return hidden[0]


hidden = encode_tokens_with_token_model(walk_tokens)

# In walk_tokens, "solar-powered rover" occupies indices 12 and 13.
# Check by looking at walk_tokens[12:14].
i, j = 12, 13

# A simple span representation concatenates:
#   start-token vector + end-token vector + mean vector across the full span.
# A span classifier could feed this vector into a linear layer to predict its type.
# `torch.cat` joins three vectors end-to-end into one longer vector.
span_vector = torch.cat([
    hidden[i],
    hidden[j],
    hidden[i:j + 1].mean(dim=0),
])

print("Hidden size:", hidden.shape[-1])
print("Span vector shape:", tuple(span_vector.shape))
print("Expected size = 3 × hidden size")

#### Why BIO can be awkward for overlapping entities

Suppose we want both:

- `[University of Nottingham Ningbo China]ORG`
- `[Ningbo]LOCATION`

The token `Ningbo` belongs to two mentions. A one-label-per-token BIO sequence cannot naturally encode both simultaneously, whereas a span model can output both spans.

In [ ]:
# Cell A2.3 — The two overlapping spans can coexist in a span representation

# GOAL: show that a span-based method can keep overlapping entities.
# Each tuple is (first_token_index, last_token_index, entity_type).
# Notice that token 3 ("Ningbo") belongs to both example spans.


nested_tokens = [
    "University", "of", "Nottingham", "Ningbo", "China"
]

# A traditional single BIO label per token cannot express both of these at once.
nested_entities = [
    (0, 4, "ORG"),
    (3, 3, "LOCATION"),
]

for start, end, typ in nested_entities:
    print(
        f"{typ:8s}:",
        " ".join(nested_tokens[start:end + 1]),
        f"(tokens {start}..{end})"
    )

### A2.1 NER is only one kind of entity extraction

Lecture 5 distinguishes **traditional named entities** from **domain-specific entities**.

Traditional NER might extract:

```text
Lin Mei  → PERSON
UNNC     → ORGANISATION
Ningbo   → LOCATION
18 May   → DATE
```

But an application-specific extractor may also need:

```text
solar-powered rover      → PROJECT
lightweight vision model → METHOD
marked paths             → ENVIRONMENT_FEATURE
```

The important lesson is that **entity extraction is broader than NER**. The ontology should be defined by the downstream task.

The following span representation shows how domain-specific mentions can be represented directly without forcing them into a traditional NER label set.


In [ ]:
# Cell A2.1 — Domain-specific spans from the Lecture 5 running example

# GOAL: demonstrate task-specific entity types beyond PERSON/ORG/DATE.
# "METHOD" and "ENVIRONMENT_FEATURE" are labels chosen for this application.


domain_tokens = [
    "The", "project", "lead", ",", "Lin", "Mei", ",", "said", "the", "rover",
    "uses", "a", "lightweight", "vision", "model", "to", "follow", "marked", "paths", "."
]

domain_entities = [
    (4, 5, "PER"),                  # Lin Mei
    (12, 14, "METHOD"),             # lightweight vision model
    (17, 18, "ENVIRONMENT_FEATURE") # marked paths
]

for start, end, entity_type in domain_entities:
    # `end + 1` ensures that the final token is included in the slice.
    text = " ".join(domain_tokens[start:end + 1])
    print(
        f"{text:30s} -> {entity_type:20s} "
        f"(tokens {start}..{end})"
    )

print(
    "\nNotice: METHOD and ENVIRONMENT_FEATURE are domain-specific entity types, "
    "not standard NER categories."
)


### A3. Recognition, typing and linking are different decisions

These three tasks answer different questions:

| Stage | Question | Example |
|---|---|---|
| Entity recognition | **Where is the mention?** | tokens 0–1 = `Nova Robotics` |
| Entity typing | **What kind of entity is it?** | `ORG` |
| Entity linking | **Which real/canonical entity does it refer to?** | `nova_001` |

BIO tagging mainly combines the first two decisions: **boundary + type**.

Entity linking comes afterwards and maps a mention to a canonical identity. Different surface forms can therefore refer to the same entity, for example `UNNC` and `University of Nottingham Ningbo China`.


In [ ]:
# Cell A3.1 — A tiny alias-based linker

# GOAL: connect a mention in text to an identity in a knowledge base.
# A dictionary maps stable IDs (keys) to entity information (values).
# An "alias" is another written name for the same real-world entity.


knowledge_base = {
    "unnc_001": {
        "canonical_name": "University of Nottingham Ningbo China",
        "aliases": {
            "UNNC",
            "University of Nottingham Ningbo China",
        },
    },
    "nova_001": {
        "canonical_name": "Nova Robotics",
        "aliases": {"Nova Robotics"},
    },
}

def link_entity(mention, kb):
    # A mention can, in principle, match multiple candidates.
    # Real entity linking therefore needs context to disambiguate candidates.
    matches = []

    # `.items()` gives both the ID and the associated information.
    for entity_id, entry in kb.items():
        # This teaching linker uses exact alias matching only.
        if mention in entry["aliases"]:
            matches.append(
                (entity_id, entry["canonical_name"])
            )

    # An empty list means no alias matched; multiple entries mean ambiguity.
    return matches


print("UNNC ->", link_entity("UNNC", knowledge_base))
print(
    "University of Nottingham Ningbo China ->",
    link_entity(
        "University of Nottingham Ningbo China",
        knowledge_base
    ),
)

This toy linker is only an alias lookup. Real entity linking must often resolve ambiguity using **mention context + candidate knowledge**.

### A4. Relation extraction as classification over ordered entity pairs

After entities are available, we can form candidate pairs and ask:

> **What relation, if any, holds between HEAD and TAIL?**

We use explicit entity markers:

```text
<HEAD> Lin Mei </HEAD> leads <TAIL> Nova Robotics </TAIL>
```

The pair is **ordered**. Swapping HEAD and TAIL changes the question.

Our relation label set is:

- `leads`
- `built`
- `demonstrated_at`
- `no_relation`

To make this classroom experiment generalise to **unseen entity names**, the model does not learn from the names themselves. During relation classification, text inside `<HEAD>...</HEAD>` and `<TAIL>...</TAIL>` is normalised to a generic `<ENTITY>` token.

For example:

```text
<HEAD> Harbour AI </HEAD> built the <TAIL> EcoSkimmer </TAIL>
```

is represented structurally as:

```text
<HEAD> <ENTITY> </HEAD> built the <TAIL> <ENTITY> </TAIL>
```

This keeps the teaching focus on **relation words and argument direction**, rather than memorising entity surface forms.

In [ ]:
# Cell A4.1 — Relation training examples with entity markers

# GOAL: give a relation classifier examples of ordered entity pairs.
# <HEAD>... </HEAD> identifies the subject of the requested relation.
# <TAIL>... </TAIL> identifies its object; swapping them changes the question.
# Each example includes both marked tokens and the correct relation label.


# Entity markers explicitly tell the model which ordered pair is being
# classified.  The same sentence can produce different relation inputs when
# HEAD and TAIL are swapped.
relation_examples = [
    (
        ["<HEAD>","Lin","Mei","</HEAD>","leads",
         "<TAIL>","Nova","Robotics","</TAIL>"],
        "leads"
    ),
    (
        ["<HEAD>","Maya","Chen","</HEAD>","leads",
         "<TAIL>","Aero","Lab","</TAIL>"],
        "leads"
    ),
    (
        ["<HEAD>","Ethan","Zhou","</HEAD>","leads",
         "<TAIL>","Smart","Farming","Lab","</TAIL>"],
        "leads"
    ),
    (
        ["<TAIL>","Lin","Mei","</TAIL>","leads",
         "<HEAD>","Nova","Robotics","</HEAD>"],
        "no_relation"
    ),
    (
        ["<TAIL>","Maya","Chen","</TAIL>","leads",
         "<HEAD>","Aero","Lab","</HEAD>"],
        "no_relation"
    ),
    (
        ["<HEAD>","Nova","Robotics","</HEAD>","built","a",
         "<TAIL>","solar-powered","rover","</TAIL>"],
        "built"
    ),
    (
        ["<HEAD>","Aero","Lab","</HEAD>","built","a",
         "<TAIL>","delivery","drone","</TAIL>"],
        "built"
    ),
    (
        ["<HEAD>","Robotics","Club","</HEAD>","built","the",
         "<TAIL>","inspection","robot","</TAIL>"],
        "built"
    ),
    (
        ["<TAIL>","Nova","Robotics","</TAIL>","built","a",
         "<HEAD>","solar-powered","rover","</HEAD>"],
        "no_relation"
    ),
    (
        ["<HEAD>","solar-powered","rover","</HEAD>","was","demonstrated","at",
         "<TAIL>","Innovation","Showcase","</TAIL>"],
        "demonstrated_at"
    ),
    (
        ["<HEAD>","electric","kart","</HEAD>","was","demonstrated","at",
         "<TAIL>","Innovation","Fair","</TAIL>"],
        "demonstrated_at"
    ),
    (
        ["<HEAD>","medical","chatbot","</HEAD>","was","demonstrated","at",
         "<TAIL>","Student","Showcase","</TAIL>"],
        "demonstrated_at"
    ),
    (
        ["<TAIL>","solar-powered","rover","</TAIL>","was","demonstrated","at",
         "<HEAD>","Innovation","Showcase","</HEAD>"],
        "no_relation"
    ),
    (
        ["<HEAD>","Lin","Mei","</HEAD>","visited",
         "<TAIL>","Aero","Lab","</TAIL>"],
        "no_relation"
    ),
    (
        ["<HEAD>","Nova","Robotics","</HEAD>","met",
         "<TAIL>","Ethan","Zhou","</TAIL>"],
        "no_relation"
    ),
]

# "no_relation" is important: not every candidate entity pair has a valid link.
REL_LABELS = [
    "leads",
    "built",
    "demonstrated_at",
    "no_relation",
]

rel_to_id = {r: i for i, r in enumerate(REL_LABELS)}
id_to_rel = {i: r for r, i in rel_to_id.items()}

print("Relation examples:", len(relation_examples))

In [ ]:
# Cell A4.2 — Relation vocabulary and batching

# GOAL: prepare relation examples as numeric inputs for the model.
# We replace specific names with <ENTITY> so the model focuses on sentence
# wording and HEAD/TAIL roles rather than memorising training-set names.


REL_SPECIAL = [
    "<PAD>",
    "<UNK>",
    "<CLS>",
    "<ENTITY>",
]

# For this teaching classifier, entity SURFACE FORMS are deliberately
# normalised away. The classifier should learn from:
#   - the relation words, and
#   - whether the marked entity is HEAD or TAIL,
# rather than memorising names seen during training.
#
# Example:
# <HEAD> Nova Robotics </HEAD> built a <TAIL> rover </TAIL>
# becomes
# <HEAD> <ENTITY> </HEAD> built a <TAIL> <ENTITY> </TAIL>
def normalise_marked_entities(tokens):
    normalised = []
    # These two Boolean flags track whether we are between entity markers
    # and whether we have already inserted <ENTITY> for this mention.
    inside_entity = False
    entity_emitted = False

    for token in tokens:
        if token in {"<HEAD>", "<TAIL>"}:
            normalised.append(token)
            inside_entity = True
            entity_emitted = False
            continue  # Do not process the same marker as normal text.

        if token in {"</HEAD>", "</TAIL>"}:
            normalised.append(token)
            inside_entity = False
            entity_emitted = False
            continue

        # Replace all the name tokens inside the markers with ONE <ENTITY>.
        if inside_entity:
            if not entity_emitted:
                normalised.append("<ENTITY>")
                entity_emitted = True
            continue

        normalised.append(token)

    return normalised


normalised_relation_examples = [
    (
        normalise_marked_entities(sentence),
        relation,
    )
    for sentence, relation in relation_examples
]

rel_vocab = REL_SPECIAL + sorted({
    token
    for sentence, _ in normalised_relation_examples
    for token in sentence
    if token not in REL_SPECIAL
})

rel_token_to_id = {
    token: i
    for i, token in enumerate(rel_vocab)
}

REL_PAD_ID = rel_token_to_id["<PAD>"]
REL_UNK_ID = rel_token_to_id["<UNK>"]
REL_CLS_ID = rel_token_to_id["<CLS>"]

REL_MAX_LEN = max(
    len(sentence) + 1
    for sentence, _ in normalised_relation_examples
)

def make_relation_batch(examples):
    input_ids, masks, y = [], [], []

    # Here each row has one relation label, unlike the BIO task (one per word).
    for tokens, relation in examples:
        tokens = normalise_marked_entities(tokens)

        # Prefix <CLS> as a start marker; unseen words become <UNK>.
        ids = [REL_CLS_ID] + [
            rel_token_to_id.get(tok, REL_UNK_ID)
            for tok in tokens
        ]

        pad_n = REL_MAX_LEN - len(ids)

        input_ids.append(
            ids + [REL_PAD_ID] * pad_n
        )
        masks.append(
            [1] * len(ids) + [0] * pad_n
        )
        y.append(rel_to_id[relation])

    # Output shapes: X_rel and M_rel = [examples, tokens]; Y_rel = [examples].
    return (
        torch.tensor(
            input_ids,
            dtype=torch.long,
            device=device,
        ),
        torch.tensor(
            masks,
            dtype=torch.bool,
            device=device,
        ),
        torch.tensor(
            y,
            dtype=torch.long,
            device=device,
        ),
    )


X_rel, M_rel, Y_rel = make_relation_batch(
    relation_examples
)

print(
    "Example before normalisation:",
    relation_examples[0][0],
)

print(
    "Example after normalisation: ",
    normalise_marked_entities(
        relation_examples[0][0]
    ),
)

print(
    X_rel.shape,
    Y_rel.shape,
)

In [ ]:
# Cell A4.3 — BiLSTM relation classifier (ordered entity-pair representation)

# GOAL: predict one relation for one ordered pair of entities.
# The model reads the words, locates both entity markers, and combines
# the corresponding vectors with a summary of the sentence context.


class TinyRelationClassifier(nn.Module):
    def __init__(self, vocab_size, num_relations, embedding_dim=48, hidden_size=40):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=REL_PAD_ID)
        self.encoder = nn.LSTM(embedding_dim, hidden_size, batch_first=True,
                               bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(6 * hidden_size, 64), nn.ReLU(), nn.Linear(64, num_relations))

    def forward(self, input_ids, attention_mask):
        # Word IDs -> embedding vectors -> contextual BiLSTM vectors.
        emb = self.token_embedding(input_ids)
        lengths = attention_mask.long().sum(dim=1).cpu().clamp(min=1)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths, batch_first=True, enforce_sorted=False)
        packed_hidden, _ = self.encoder(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(
            packed_hidden, batch_first=True, total_length=input_ids.size(1))
        # Entity markers delimit each normalised mention. After normalisation,
        # each entity surface is represented by the token <ENTITY>.
        head_marker = rel_token_to_id["<HEAD>"]
        tail_marker = rel_token_to_id["<TAIL>"]
        batch_indices = torch.arange(input_ids.size(0), device=input_ids.device)
        # Find the token immediately AFTER the <HEAD> or <TAIL> marker.
        # After normalisation this position contains <ENTITY>.
        head_pos = (input_ids == head_marker).int().argmax(dim=1) + 1
        tail_pos = (input_ids == tail_marker).int().argmax(dim=1) + 1
        head_repr = h[batch_indices, head_pos.clamp(max=h.size(1)-1)]
        tail_repr = h[batch_indices, tail_pos.clamp(max=h.size(1)-1)]
        # Average only real-token vectors to summarise the sentence.
        # `unsqueeze` adds an axis so the mask can multiply each vector.
        context = (h * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1).clamp(min=1).unsqueeze(-1)
        # Three BiLSTM vectors, each of width 2*hidden_size, give 6*hidden_size.
        # Their ORDER matters because relations may be directional.
        ordered_pair = torch.cat((head_repr, tail_repr, context), dim=-1)
        return self.classifier(ordered_pair)


relation_model = TinyRelationClassifier(vocab_size=len(rel_vocab), num_relations=len(REL_LABELS)).to(device)

In [ ]:
# Cell A4.4 — Train the relation classifier

# GOAL: train the relation classifier in the same way as the BIO tagger.
# Unlike token tagging, each example here has ONE answer for the whole pair.


set_seed()

rel_optimizer = torch.optim.Adam(
    relation_model.parameters(),
    lr=0.01,
)
rel_criterion = nn.CrossEntropyLoss()

relation_train_start = time.perf_counter()

for epoch in range(151):
    relation_model.train()
    rel_optimizer.zero_grad()

    # One class prediction is produced for each ordered entity pair.
    logits = relation_model(X_rel, M_rel)
    # Compare each pair's predicted relation with the correct relation ID.
    loss = rel_criterion(logits, Y_rel)

    # Compute gradients and use Adam to improve the model.
    loss.backward()
    torch.nn.utils.clip_grad_norm_(relation_model.parameters(), max_norm=1.0)
    rel_optimizer.step()

    if epoch % 50 == 0 or epoch == 150:
        print(f"epoch={epoch:3d} | loss={loss.item():.4f}")

relation_train_seconds = time.perf_counter() - relation_train_start
print(f"\nRelation-classifier training time: {relation_train_seconds:.2f} seconds")


In [ ]:
# Cell A4.5 — Prediction helper

# GOAL: classify a NEW sentence with <HEAD> and <TAIL> markers.
# This helper returns both the predicted relation and its probability table.


def predict_relation(marked_tokens):
    # Apply the SAME entity-surface normalisation used during training.
    normalised_tokens = normalise_marked_entities(
        marked_tokens
    )

    ids = [REL_CLS_ID] + [
        rel_token_to_id.get(tok, REL_UNK_ID)
        for tok in normalised_tokens
    ]

    x = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device,
    )

    mask = torch.ones_like(
        x,
        dtype=torch.bool,
    )

    # Set inference mode: we are predicting, not training.
    relation_model.eval()

    with torch.no_grad():
        logits = relation_model(
            x,
            mask,
        )

        # Softmax converts raw scores into probabilities that sum to 1.
        probs = torch.softmax(
            logits,
            dim=-1,
        )[0]

    # Choose the relation with the largest predicted probability.
    best_id = int(
        probs.argmax().cpu()
    )

    # Dictionary comprehension: one human-readable probability per relation.
    probability_table = {
        id_to_rel[i]: float(
            probs[i].cpu()
        )
        for i in range(
            len(REL_LABELS)
        )
    }

    return (
        id_to_rel[best_id],
        probability_table,
    )

In [ ]:
# Cell A4.6 — Direction matters

# GOAL: test why entity order matters when extracting directed relations.
# We keep the words the same but reverse which mention is HEAD and TAIL.


normal_pair = [
    "<HEAD>", "Lin", "Mei", "</HEAD>",
    "leads",
    "<TAIL>", "Nova", "Robotics", "</TAIL>",
]

# Reversing roles does NOT reverse the sentence; only markers change.
reversed_pair = [
    "<TAIL>", "Lin", "Mei", "</TAIL>",
    "leads",
    "<HEAD>", "Nova", "Robotics", "</HEAD>",
]

normal_pred, normal_probs = predict_relation(normal_pair)
reverse_pred, reverse_probs = predict_relation(reversed_pair)

print("HEAD=Lin Mei, TAIL=Nova Robotics")
print("prediction:", normal_pred)
# `round(v, 3)` displays each probability to three decimal places.
print({k: round(v, 3) for k, v in normal_probs.items()})

print("\nHEAD=Nova Robotics, TAIL=Lin Mei")
print("prediction:", reverse_pred)
print({k: round(v, 3) for k, v in reverse_probs.items()})

### A4.1 Pipeline extraction and error propagation

A common **pipeline** is:

```text
text
  ↓
entity extractor
  ↓
candidate entity pairs
  ↓
relation classifier
  ↓
relation tuples
```

This is modular and easy to debug, but a relation can only be predicted if its entities were found first.

For example, if the entity extractor misses `solar-powered rover`, the candidate pair

```text
(Nova Robotics, solar-powered rover)
```

is never created, so the relation

```text
built(Nova Robotics, solar-powered rover)
```

cannot be recovered by the later relation classifier.

A **joint extraction** model instead predicts entities and relations in a shared process, allowing evidence from the two tasks to interact. Joint models can reduce some pipeline errors, but they are usually more complex to train and debug.


In [ ]:
# Cell A4.1 — Simulate pipeline error propagation

# GOAL: simulate how an entity-recognition mistake affects later stages.
# A relation classifier cannot assess pairs that the entity extractor misses.


def all_ordered_entity_pairs(entities):
    """Return every ordered pair of distinct extracted entity strings."""
    # A nested comprehension considers both (A,B) and (B,A), not (A,A).
    return [
        (head, tail)
        for head in entities
        for tail in entities
        if head != tail
    ]

gold_entities_pipeline = [
    "Lin Mei",
    "Nova Robotics",
    "solar-powered rover",
]

# The gold list contains 3 entities; the predicted list only 2.
# Simulate an entity extractor that MISSES the project mention.
predicted_entities_pipeline = [
    "Lin Mei",
    "Nova Robotics",
]

gold_pairs = all_ordered_entity_pairs(gold_entities_pipeline)
predicted_pairs = all_ordered_entity_pairs(predicted_entities_pipeline)

target_pair = ("Nova Robotics", "solar-powered rover")

print("Target relation pair:", target_pair)
print("Available with gold entities?     ", target_pair in gold_pairs)
print("Available after entity miss?      ", target_pair in predicted_pairs)

print(
    "\nPipeline consequence: if a required entity is missing, "
    "some downstream relation candidates disappear completely."
)


### A5. From pairwise relations to an event frame

Relation extraction connects two entities at a time. Event extraction groups several pieces of information around one **occurrence**.

For:

> Nova Robotics will **demonstrate** a solar-powered rover at Innovation Showcase on 18 May.

we represent:

```text
trigger = demonstrate
event type = Demonstration

Agent    -> Nova Robotics
Artifact -> solar-powered rover
Venue    -> Innovation Showcase
Date     -> 18 May
```

The **trigger** answers *what happened?*  
The **arguments** answer *who/what/where/when participated in that event?*


In [ ]:
# Cell A5.1 — Event frame

# GOAL: store the participants in one event, not just separate pairwise facts.
# A Python dictionary uses field names (keys) such as "trigger" and "arguments".
# `json.dumps(..., indent=2)` prints a neatly formatted version.


# An event frame groups one trigger with typed semantic-role arguments.
event_frame = {
    "trigger": "demonstrate",
    "event_type": "Demonstration",
    # Each role connects the event to an appropriate extracted mention.
    "arguments": {
        "Agent": "Nova Robotics",
        "Artifact": "solar-powered rover",
        "Venue": "Ningbo Campus Innovation Showcase",
        "Date": "18 May",
    },
}

print(json.dumps(event_frame, indent=2))

### A5.1 A sentence can contain more than one event

Lecture 5 uses the following example:

> Lin Mei **announced** that Nova Robotics will **demonstrate** its solar rover in the student innovation area on 18 May.

There are two triggers:

- `announced` → `Announcement`
- `demonstrate` → `Demonstration`

The difficult part is not merely finding verbs. The extractor must decide **which arguments attach to which event**. For example, `18 May` belongs to the demonstration event, not necessarily to the announcement event.


In [ ]:
# Cell A5.1 — Two event frames from one sentence

# GOAL: represent two different events mentioned in the same sentence.
# This is a LIST of dictionaries: each dictionary describes one event.
# Python None becomes JSON null and means the value is not given.


# Distinguish the act of announcing from the future demonstration itself.
multiple_event_frames = [
    {
        "trigger": "announced",
        "event_type": "Announcement",
        "arguments": {
            "Speaker": "Lin Mei",
            "Content": "demonstration plan",
            "Location": None,
            "Date": None,
        },
    },
    {
        "trigger": "demonstrate",
        "event_type": "Demonstration",
        "arguments": {
            "Agent": "Nova Robotics",
            "Artifact": "solar rover",
            "Location": "student innovation area",
            "Date": "18 May",
        },
    },
]

print(json.dumps(multiple_event_frames, indent=2))

print(
    "\nThe null values are meaningful: the source sentence does not specify "
    "a separate announcement location or announcement date."
)


### A5.2 A schema is a contract

Revised Lecture 5 treats every extracted record as a **candidate** that must be checked before it is stored:

1. **Syntax:** can the JSON representation be parsed?
2. **Schema:** are required fields, types and permitted values correct?
3. **Evidence:** are non-null extracted values supported by the source text?

A valid JSON object may still contain an unsupported claim or the wrong field type.

In [ ]:
# Cell A5.2 — Record schema and validator

# GOAL: prevent incorrect records from entering downstream software.
# The SCHEMA describes what a valid record is allowed to contain.
# The VALIDATOR checks both the record format and support from source text.
# This is a simple illustration; substring matching is not full factual checking.


source_text_A = (
    "Nova Robotics, a student team from UNNC, will demonstrate "
    "a solar-powered rover at the Ningbo Campus Innovation Showcase on 18 May. "
    "The project lead, Lin Mei, said the rover uses a lightweight vision model "
    "to follow marked paths."
)

# The schema is a CONTRACT between the extractor and downstream software.
#
# `required` answers: must the FIELD be present?
# `nullable` answers: is None an acceptable value when the source omits it?
# str = Python text type; required=True means the key must exist;
# nullable=True means its value may be None if the source is silent.
schema_A = {
    "team": {
        "type": str,
        "required": True,
        "nullable": False,
    },
    "project": {
        "type": str,
        "required": True,
        "nullable": False,
    },
    "event_type": {
        "type": str,
        "required": True,
        "nullable": False,
        "allowed": {"Demonstration"},
    },
    "venue": {
        "type": str,
        "required": True,
        "nullable": True,
    },
    "date": {
        "type": str,
        "required": True,
        "nullable": True,
    },
    "project_lead": {
        "type": str,
        "required": True,
        "nullable": True,
    },
}

candidate_record_A = {
    "team": "Nova Robotics",
    "project": "solar-powered rover",
    "event_type": "Demonstration",
    "venue": "Ningbo Campus Innovation Showcase",
    "date": "18 May",
    "project_lead": "Lin Mei",
}

EVENT_TRIGGER_EVIDENCE = {
    "Demonstration": {
        "demonstrate",
        "demonstrated",
        "demo",
        "demoed",
    }
}

def validate_record(record, source_text, schema):
    """Check schema constraints and simple source grounding.

    This teaching validator separates:
      1. schema validation — is the structure allowed?
      2. evidence validation — are extracted values supported by the source?
    """
    errors = []

    # ------------------------------------------------------------
    # 1. SCHEMA VALIDATION
    # ------------------------------------------------------------
    # A set holds unique field names. Set difference (-) finds unexpected keys.
    allowed_keys = set(schema)
    extra_keys = set(record) - allowed_keys

    if extra_keys:
        errors.append(
            f"schema error: unexpected fields {sorted(extra_keys)}"
        )

    # Loop over every expected field and its rules (specification).
    for field, spec in schema.items():
        if field not in record:
            if spec.get("required", False):
                errors.append(
                    f"schema error: missing required field '{field}'"
                )
            continue

        value = record[field]

        # None can be a VALID extraction when the schema explicitly allows it.
        # This is safer than inventing a value that the text never supplied.
        if value is None:
            if not spec.get("nullable", False):
                errors.append(
                    f"schema error: field '{field}' may not be null"
                )
            continue

        # `isinstance` checks whether a value has the expected Python type.
        if not isinstance(value, spec["type"]):
            errors.append(
                f"schema error: '{field}' has wrong type "
                f"{type(value).__name__}"
            )
            continue

        if "allowed" in spec and value not in spec["allowed"]:
            errors.append(
                f"schema error: '{field}' has disallowed value {value!r}"
            )

    # ------------------------------------------------------------
    # 2. EVIDENCE / GROUNDING VALIDATION
    # ------------------------------------------------------------
    # Parseable JSON and schema compliance are not enough.
    # Extracted field values must also be supported by the source text.
    # Lowercase both sides so simple letter-case differences do not matter.
    # LIMITATION: literal substring checks miss paraphrases and do not prove
    # the correct relationship between entities; this is a teaching validator.
    source_lower = source_text.lower()

    for field in [
        "team",
        "project",
        "venue",
        "date",
        "project_lead",
    ]:
        value = record.get(field)

        if isinstance(value, str) and value.lower() not in source_lower:
            errors.append(
                f"evidence error: {field}={value!r} "
                "is not supported by a literal source span"
            )

    # event_type is an abstract label rather than a literal field value.
    # We therefore look for a supporting trigger instead of literal equality.
    event_type = record.get("event_type")

    if isinstance(event_type, str):
        triggers = EVENT_TRIGGER_EVIDENCE.get(event_type, set())

        # `any` is True when at least one permitted event word appears.
        if triggers and not any(
            trigger in source_lower
            for trigger in triggers
        ):
            errors.append(
                f"evidence error: event_type={event_type!r} "
                "has no recognised trigger in the text"
            )

    # An empty list [] means that NONE of these checks found a problem.
    return errors


print(json.dumps(candidate_record_A, indent=2))
print("\nValidation errors:")
print(
    validate_record(
        candidate_record_A,
        source_text_A,
        schema_A,
    )
)


In [ ]:
# Cell A5.2 — Inject two common failures

# GOAL: test whether the validator catches plausible-looking mistakes.
# One value is invented, and one field is not permitted by the schema.


bad_record_A = {
    "team": "Nova Robotics",
    "project": "solar-powered rover",
    "event_type": "Demonstration",
    "venue": "Ningbo Campus Innovation Showcase",
    "date": "18 May",
    "project_lead": "Daniel Wu",  # unsupported generation
    "organisation": "UNNC",        # not allowed by this schema
}

# The for-loop prints one helpful message for each validation failure.
for error in validate_record(
    bad_record_A,
    source_text_A,
    schema_A,
):
    print("•", error)

### A5.3 Missing information should be represented explicitly

If the source says:

> Nova Robotics will demonstrate a solar rover.

then there is **no date** in the source.

A robust schema can allow:

```json
"date": null
```

This is better than guessing `"18 May"`.

Nullable fields reduce pressure to hallucinate missing values.


In [ ]:
# Cell A5.3 — Null is different from an invented value

# GOAL: distinguish "the source did not tell us" from "we guessed".
# `None` (shown as null in JSON) is acceptable for nullable missing fields.


source_without_date = (
    "Nova Robotics will demonstrate a solar-powered rover."
)

record_with_null = {
    "team": "Nova Robotics",
    "project": "solar-powered rover",
    "event_type": "Demonstration",
    "venue": None,
    "date": None,
    "project_lead": None,
}

# This example guesses a date even though the sentence contains no date.
record_with_invented_date = {
    "team": "Nova Robotics",
    "project": "solar-powered rover",
    "event_type": "Demonstration",
    "venue": None,
    "date": "18 May",       # unsupported by this source
    "project_lead": None,
}

print("Record using null:")
print(validate_record(
    record_with_null,
    source_without_date,
    schema_A,
))

print("\nRecord inventing a date:")
print(validate_record(
    record_with_invented_date,
    source_without_date,
    schema_A,
))


### A5.4 A schema must preserve repeated structure

Suppose a document describes two team–project pairs:

```text
Nova Robotics → solar rover
GreenAI Club  → energy-monitoring dashboard
```

A poor schema that stores all team names in one string and all projects in another string loses the **pairing**.

A better schema uses an array of repeated records:

```json
{
  "records": [
    {"team": "Nova Robotics", "project": "solar rover"},
    {"team": "GreenAI Club", "project": "energy-monitoring dashboard"}
  ]
}
```

Schema design is therefore part of the extraction problem, not merely formatting.


In [ ]:
# Cell A5.4 — Repeated records preserve relationships

# GOAL: keep each team correctly paired with its own project.
# A single long text string loses which team built which project.
# A list of small records preserves each team-project relationship.


bad_flat_record = {
    "teams": "Nova Robotics and GreenAI Club",
    "projects": "solar rover and energy-monitoring dashboard",
}

# The "records" list preserves individual team-project pairs.
better_repeated_record = {
    "records": [
        {
            "team": "Nova Robotics",
            "project": "solar rover",
        },
        {
            "team": "GreenAI Club",
            "project": "energy-monitoring dashboard",
        },
    ]
}

print("Poor schema:")
print(json.dumps(bad_flat_record, indent=2))

print("\nBetter repeated-record schema:")
print(json.dumps(better_repeated_record, indent=2))


### A5.5 Validate a structured record before storage

In revised Lecture 5, extraction uses the sequence of processing stages:

```text
source text → entities → ordered relations / events → candidate record
                                                    ↓
                                   parse → schema check → evidence check
                                                    ↓
                                          accept or correct
```

Parsing, schema validity and source grounding are **different conditions**. Here we practise an automated gate using a candidate JSON string. This is ordinary record validation: **no LLM or Transformer is needed**.

In [ ]:
# Cell A5.5 — Parse -> validate -> accept or retry

# GOAL: accept an extracted record only after several checks.
# First convert JSON text to a Python dictionary; then validate its content.
# `try` / `except` prevents invalid JSON from crashing the notebook.


candidate_json_text = """
{
  "team": "Nova Robotics",
  "project": "solar-powered rover",
  "event_type": "Demonstration",
  "venue": "Ningbo Campus Innovation Showcase",
  "date": "18 May",
  "project_lead": "Lin Mei"
}
"""

def parse_and_validate_candidate(json_text, source_text, schema):
    """A minimal version of a professional structured-extraction gate."""
    try:
        # Stage 1: syntax / parsing.
        # `json.loads` parses a JSON STRING into Python objects.
        record = json.loads(json_text)
    except json.JSONDecodeError as exc:
        return None, [f"JSON parse error: {exc.msg}"]

    # We expect the outermost JSON value to have named fields (a dictionary).
    if not isinstance(record, dict):
        return None, ["schema error: top-level JSON value must be an object"]

    # Stages 2 and 3: schema + evidence checks.
    errors = validate_record(record, source_text, schema)

    return record, errors


parsed_record, gate_errors = parse_and_validate_candidate(
    candidate_json_text,
    source_text_A,
    schema_A,
)

print("Parsed candidate:")
print(parsed_record)

print("\nGate result:")
# Non-empty error list -> reject for correction; no errors -> accept for review.
if gate_errors:
    print("CORRECT RECORD BEFORE STORAGE")
    for error in gate_errors:
        print("•", error)
else:
    print("ACCEPT FOR STORAGE / REVIEW")


### A5.6 Pipeline stages versus validation checks

| Stage | Main input | Output | Common failure |
|---|---|---|---|
| BIO / span extraction | tokens | entity spans and types | wrong boundary or label |
| Entity linking | mentions + knowledge base | canonical identity | ambiguous alias |
| Relation classification | sentence + ordered pair | relation label | direction error |
| Event extraction | context + entities | trigger and role arguments | missing participant |
| Record construction | extracted facts + schema | structured candidate | missing or mismatched fields |
| Record validation | candidate + source | accept / errors | unsupported value |

A correct *format* is necessary but not sufficient for a correct extracted fact.

### A6. Structured relations can answer a graph-style question

Suppose the extracted graph contains:

```text
Lin Mei --leads--> Nova Robotics
Nova Robotics --built--> solar-powered rover
solar-powered rover --demonstrated_at--> Innovation Showcase
```

A downstream system can follow edges rather than reread the original sentence.

In [ ]:
# Cell A6.1 — Tiny graph reasoning without any graph library

# GOAL: answer a question by following two links in a tiny graph.
# A triple is (subject, relation, object), such as (team, built, project).
# We first find the builder, then find who leads that builder.


relation_tuples_A = [
    ("Lin Mei", "leads", "Nova Robotics"),
    ("Nova Robotics", "built", "solar-powered rover"),
    ("solar-powered rover", "demonstrated_at", "Ningbo Campus Innovation Showcase"),
]

def who_leads_team_that_built(project, triples):
    # Step 1: find every entity that has a `built` edge to the project.
    # A set comprehension collects every team that built this project.
    builders = {
        head
        for head, relation, tail in triples
        if relation == "built" and tail == project
    }

    # Step 2: find every person with a `leads` edge to one of those builders.
    # Follow a second edge: person --leads--> any team we just found.
    leaders = {
        head
        for head, relation, tail in triples
        if relation == "leads" and tail in builders
    }

    # `sorted` produces an easy-to-read, consistent list of names.
    return sorted(leaders)


answer = who_leads_team_that_built(
    "solar-powered rover",
    relation_tuples_A,
)

print("Who leads the team that built the solar-powered rover?")
print(answer)

### A7. Runtime check

The neural models in this lab are intentionally small. The required experiments should run comfortably on CPU.

The timings below measure only the two model-training stages, not time spent reading or completing exercises.


In [ ]:
# Cell A7.1 — Core neural-training runtime

# GOAL: check whether both training stages fit within class time.
# Times are in seconds; `15 * 60` means 15 minutes in seconds.


# Add the two measured durations; this does not include notebook reading time.
core_training_seconds = token_train_seconds + relation_train_seconds

print(f"Token classifier:    {token_train_seconds:7.2f} s")
print(f"Relation classifier: {relation_train_seconds:7.2f} s")
print("-" * 34)
print(f"Core neural training:{core_training_seconds:7.2f} s")

if core_training_seconds < 15 * 60:
    print("\n✓ Core training is below the 15-minute classroom target.")
else:
    print("\nWarning: runtime exceeded 15 minutes on this machine.")


### A8. Evaluation begins by choosing the output unit

Different IE tasks produce different objects, so their correctness rules differ.

| Task | Output unit | What must match? |
|---|---|---|
| Entity extraction | span + type | exact boundary and type |
| Relation extraction | head + relation + tail | both entities, relation label and direction |
| Event extraction | trigger + role arguments | event trigger/type and required role fillers |
| Record extraction | field–value pair | expected field and correct value |
| Structured record validation | field–value record | parseable, schema-valid and grounded |

This is why one generic “accuracy” number is not enough for every IE problem.

Also inspect **precision and recall separately**:

- low precision usually means too many false positives;
- low recall usually means too many false negatives;
- the same F1 score can hide very different failure patterns.


---

# Part B — Student exercises

## New announcement

> **Harbour AI**, a student team from **UNNC**, will demonstrate an **autonomous litter-collecting boat** at the **Green Campus Expo** on **22 September**. The project lead, **Aisha Rahman**, said the boat uses a lightweight vision model.

Use the same ideas and helper functions from Part A.

### Target entity schema for the exercises

| Mention | Type |
|---|---|
| Harbour AI | ORG |
| UNNC | ORG |
| autonomous litter-collecting boat | PROJ |
| Green Campus Expo | EVENT |
| 22 September | DATE |
| Aisha Rahman | PER |

The phrase **lightweight vision model** is deliberately *outside* this exercise's entity schema. This reminds us that “what counts as an entity” depends on the task definition.


In [ ]:
# Cell B0.1 — Pretokenised exercise text

# GOAL: inspect the NEW exercise sentence and its word positions.
# `enumerate` gives both the position (i) and the word (tok).
# Use these zero-based indices when describing an entity's boundaries.


exercise_tokens = [
    "Harbour", "AI", "a", "student", "team", "from", "UNNC",
    "will", "demonstrate", "an",
    "autonomous", "litter-collecting", "boat",
    "at", "Green", "Campus", "Expo",
    "on", "22", "September",
    "The", "project", "lead", "Aisha", "Rahman",
    "said", "the", "boat", "uses", "a",
    "lightweight", "vision", "model"
]

print("Number of tokens:", len(exercise_tokens))
# Example output "0: Harbour" means Harbour is token at index 0.
for i, tok in enumerate(exercise_tokens):
    print(f"{i:2d}: {tok}")

## Exercise B1 — Write the BIO labels

Create one BIO label for every token in `exercise_tokens`.

Requirements:

- multi-token entities must use `B-TYPE` followed by `I-TYPE`;
- `UNNC` is a one-token `ORG`;
- words outside the six target entity mentions are `O`;
- your label list must have exactly the same length as `exercise_tokens`.

Replace `None` below with your list.

In [ ]:
# Cell B1.1 — YOUR CODE HERE

# Python reminder: a list looks like ["B-ORG", "I-ORG", "O"].
# Start at token 0 and move in order; do not skip punctuation or small words.
# You are filling in the labels, NOT changing the exercise_tokens list.
# Write exactly one BIO label for every token in `exercise_tokens`.
#
# Useful checks:
# - multi-token entity: B-TYPE followed by one or more I-TYPE labels;
# - single-token entity: only B-TYPE;
# - non-entity token: O;
# - len(exercise_labels) must equal len(exercise_tokens).
#
# Example shape only:
# exercise_labels = ["B-ORG", "I-ORG", "O", ...]

# Replace None with your complete list of labels when you are ready.
# None means "not answered yet" and allows the next cell to run safely.
exercise_labels = None


In [ ]:
# Cell B1.2 — Self-check and visualise your answer

# GOAL: check your BIO answer only after you have entered it in B1.1.
# The supplied gold entities are the reference answers for automatic scoring.
# The `if` / `elif` / `else` branches handle unfinished, invalid, and ready inputs.


gold_entities_B = [
    ("Harbour AI", "ORG", 0, 1),
    ("UNNC", "ORG", 6, 6),
    ("autonomous litter-collecting boat", "PROJ", 10, 12),
    ("Green Campus Expo", "EVENT", 14, 16),
    ("22 September", "DATE", 18, 19),
    ("Aisha Rahman", "PER", 23, 24),
]

# None is a placeholder: until B1.1 is done, no answer is available.
if exercise_labels is None:
    print("B1 not completed yet.")
elif len(exercise_labels) != len(exercise_tokens):
    print(
        "Length mismatch:",
        len(exercise_labels),
        "labels for",
        len(exercise_tokens),
        "tokens"
    )
else:
    decoded_B = decode_bio(
        exercise_tokens,
        exercise_labels,
    )

    print("Decoded entities:")
    for entity in decoded_B:
        print(entity)

    print("\nExact-match score:")
    # Precision/recall/F1 require matching the COMPLETE span and category.
    print(prf1(decoded_B, gold_entities_B))

    render_entities(exercise_tokens, decoded_B)

### B1 reflection

Answer in a Markdown cell below:

1. Why is `Harbour AI` encoded `B-ORG, I-ORG` rather than `ORG, ORG`?
2. If `Green Campus Expo` were labelled `B-EVENT, I-EVENT, O`, what kind of error would that create at entity level?

**Your B1 reflection:**

1. ...
2. ...


## Exercise B2 — Candidate spans and the maximum span length

Use `candidate_spans()` from Part A.

1. Generate candidates with `max_span_length=3`.
2. Check whether **all six gold entities** can be represented.
3. Repeat with `max_span_length=2`.
4. Identify which gold entity mentions become impossible to propose.

This illustrates the trade-off:

- larger $L$ → better chance of covering long entities;
- larger $L$ → more candidate spans and more computation.

In [ ]:
# Cell B2.1 — YOUR CODE HERE

# Use the SAME token sequence but change only the maximum candidate length.
# This gives a controlled comparison of candidate coverage versus computation.

# Replace each None with a call to candidate_spans(...).
# Keep exercise_tokens unchanged so span length is the ONLY difference.
# Longer spans can cover longer mentions but create more candidates.
spans_L3 = None
spans_L2 = None

# Suggested start:
# spans_L3 = candidate_spans(exercise_tokens, max_span_length=3)
# spans_L2 = candidate_spans(exercise_tokens, max_span_length=2)

# This safety check skips printing until you have defined spans_L3.
if spans_L3 is not None:
    print("Number of L=3 candidates:", len(spans_L3))

if spans_L2 is not None:
    print("Number of L=2 candidates:", len(spans_L2))


**Your B2 answer:**

- Entities representable with maximum length 3: ...
- Entities lost with maximum length 2: ...
- Why: ...


## Exercise B3 — Test the tiny BIO tagger on unseen text

Now run the scratch-trained model from Part A on the new announcement.

**Do not expect good generalisation.** Many new words are mapped to a single `<UNK>` embedding. This is useful: it exposes why real NLP systems use better representations and more training examples.

Your tasks:

1. run `predict_bio()` below;
2. decode the predictions;
3. calculate exact entity P/R/F1 against `gold_entities_B`;
4. identify at least **two boundary/type errors**;
5. explain how additional labelled examples or more robust embeddings could help.

In [ ]:
# Cell B3.1 — Inference helper (provided)

# GOAL: use the trained tagger on an unseen exercise sentence.
# We reuse the SAME word-to-number mapping from training.
# The model will struggle with unfamiliar words mapped to <UNK>.


def predict_bio(tokens):
    # Add an outer [...] for one sentence: input shape is [1, tokens].
    ids = torch.tensor(
        [[token_to_id.get(tok, UNK_ID) for tok in tokens]],
        dtype=torch.long,
        device=device,
    )
    mask = torch.ones_like(ids, dtype=torch.bool)

    token_model.eval()

    with torch.no_grad():
        logits = token_model(ids, mask)
        # Highest-scoring class ID per token; [0] selects our only sentence.
        # `.cpu().tolist()` converts the output to an ordinary Python list.
        pred_ids = logits.argmax(dim=-1)[0].cpu().tolist()

    return [id_to_label[i] for i in pred_ids]


model_labels_B = predict_bio(exercise_tokens)

for tok, pred in zip(exercise_tokens, model_labels_B):
    print(f"{tok:20s} {pred}")

In [ ]:
# Cell B3.2 — YOUR CODE HERE

# The model has already predicted one BIO label per token in `model_labels_B`.
#
# Your job:
# 1. decode the BIO sequence into entity spans;
# 2. compare those spans with `gold_entities_B`;
# 3. inspect exact-match precision, recall and F1.

# First call decode_bio; then give the resulting spans to prf1.
# A model may label many words correctly but still miss complete entities.
model_entities_B = None
model_scores_B = None

# Suggested:
# model_entities_B = decode_bio(exercise_tokens, model_labels_B)
# model_scores_B = prf1(model_entities_B, gold_entities_B)

if model_entities_B is not None:
    print("Predicted entities:")
    for item in model_entities_B:
        print(item)

    print("\nScores:", model_scores_B)
    render_entities(exercise_tokens, model_entities_B)


**Your B3 diagnosis:**

- Boundary or type error 1: ...
- Boundary or type error 2: ...
- Why the shared `<UNK>` embedding is a limitation: ...
- One improvement based on Lecture 2 or Lecture 4: ...

## Exercise B4 — Relation extraction with entity markers

Use this mini relation text:

> Aisha Rahman leads Harbour AI. Harbour AI built the EcoSkimmer. EcoSkimmer was demonstrated at Green Campus Expo.

Construct marked token lists for these ordered pairs:

1. HEAD = `Aisha Rahman`, TAIL = `Harbour AI`
2. HEAD = `Harbour AI`, TAIL = `EcoSkimmer`
3. HEAD = `EcoSkimmer`, TAIL = `Green Campus Expo`
4. Reverse pair 1 so that HEAD = `Harbour AI`, TAIL = `Aisha Rahman`

Run `predict_relation()` on each.

Then answer: **Why should the prediction for pair 4 differ from pair 1 even though the same two entities are present?**

After testing the four marked pairs, answer one additional question:

> If the entity extractor completely misses `EcoSkimmer`, which relation candidate(s) become impossible for a downstream **pipeline** relation classifier to create?


The classifier applies the same entity-surface normalisation at prediction time. Therefore unfamiliar names such as `Harbour AI` and `EcoSkimmer` should not by themselves cause an `<UNK>`-driven relation error.

In [ ]:
# Cell B4.1 — YOUR CODE HERE

# Entity markers encode the ORDERED pair.
# HEAD and TAIL are not interchangeable for directional relations such as `leads`.

# Replace each None with a list of words and entity-marker strings.
# Place HEAD around the entity you are asking ABOUT and TAIL around
# the other entity; use the relation words shown in the exercise.
marked_pair_1 = None
marked_pair_2 = None
marked_pair_3 = None
marked_pair_4 = None

# Example syntax:
# marked_pair_1 = [
#     "<HEAD>", "...", "</HEAD>",
#     "relation words",
#     "<TAIL>", "...", "</TAIL>",
# ]

for name, pair in [
    ("pair 1", marked_pair_1),
    ("pair 2", marked_pair_2),
    ("pair 3", marked_pair_3),
    ("pair 4", marked_pair_4),
]:
    if pair is None:
        print(name, "not completed")
    else:
        # A completed pair returns a predicted label and probabilities.
        pred, probs = predict_relation(pair)
        print(
            name,
            "->",
            pred,
            {k: round(v, 3) for k, v in probs.items()}
        )


**Your B4 explanation:**

...


## Exercise B5 — Event frame and record validation

For the original Harbour AI announcement:

1. create an event frame with:
   - `trigger`
   - `event_type`
   - `Agent`
   - `Artifact`
   - `Venue`
   - `Date`
2. create a record with:
   - `team`
   - `project`
   - `event_type`
   - `venue`
   - `date`
   - `project_lead`
3. validate the record.

Start from the intentionally bad record below. It contains at least two problems.

Then consider a second source:

> Harbour AI will demonstrate an autonomous litter-collecting boat.

There is no venue, date or project lead. Explain why a well-designed schema should allow those fields to be `null` instead of encouraging the extractor to guess them.


In [ ]:
# Cell B5.1 — Source and schema

# GOAL: inspect two kinds of error before constructing your own record.
# `schema_A.copy()` reuses the existing validation rules for this new text.
# The example intentionally contains one invalid category and one invented name.


source_text_B = (
    "Harbour AI, a student team from UNNC, will demonstrate "
    "an autonomous litter-collecting boat at the Green Campus Expo "
    "on 22 September. The project lead, Aisha Rahman, said the boat "
    "uses a lightweight vision model."
)

# This makes a shallow copy; we do not change the field specifications.
schema_B = schema_A.copy()

bad_record_B = {
    "team": "Harbour AI",
    "project": "autonomous litter-collecting boat",
    "event_type": "Demo",       # schema problem
    "venue": "Green Campus Expo",
    "date": "22 September",
    "project_lead": "Daniel Wu",  # unsupported generation
}

for error in validate_record(
    bad_record_B,
    source_text_B,
    schema_B,
):
    print("•", error)

In [ ]:
# Cell B5.2 — YOUR CODE HERE

# Build TWO structures:
# 1. an event frame centred on the trigger `demonstrate`;
# 2. a schema-compliant record for downstream software.
#
# Important: use only values supported by `source_text_B`.

# Replace None with a dictionary containing the event trigger, event type
# and an "arguments" dictionary of participant roles.
event_frame_B = None

# Replace the None values below with literal mentions from source_text_B.
# Keep the given field names unchanged; the validator expects these keys.
student_record_B = {
    "team": None,
    "project": None,
    "event_type": None,
    "venue": None,
    "date": None,
    "project_lead": None,
}

print("Event frame:")
print(event_frame_B)

print("\nRecord:")
print(json.dumps(student_record_B, indent=2))

print("\nValidation errors:")
for error in validate_record(
    student_record_B,
    source_text_B,
    schema_B,
):
    print("•", error)


### B5 reflection

Why is the following statement false?

> “If the model returned valid JSON, the extraction is correct.”

Your answer should mention **schema validity** and **evidence validity**.

**Your B5 reflection:**

...


## Exercise B6 — Evaluation and error analysis

Assume the gold entity set is the six entities in `gold_entities_B`.

A different model predicts the items below.

Your tasks:

1. compute precision, recall and F1 using `prf1()`;
2. list the false positives and false negatives;
3. classify the mistakes using categories such as:
   - boundary error,
   - type error,
   - missed entity,
   - spurious/out-of-schema entity;
4. suggest one likely improvement for each major error pattern.

Remember: the same F1 score can hide different failure patterns, so inspect precision and recall separately.

In [ ]:
# Cell B6.1 — Fixed predictions for error analysis

# GOAL: calculate which complete entities were predicted wrongly or missed.
# A false positive (FP) is a predicted entity not in the gold set.
# A false negative (FN) is a gold entity not found in the prediction set.


# Each tuple is (text, category, start_token, end_token).
# A wrong boundary/type is a different tuple, even if the text is similar.
error_analysis_predictions_B = [
    ("Harbour AI", "ORG", 0, 1),                         # correct
    ("UNNC", "ORG", 6, 6),                              # correct
    ("litter-collecting boat", "PROJ", 11, 12),         # boundary error
    ("Green Campus Expo", "ORG", 14, 16),               # type error
    ("22 September", "DATE", 18, 19),                   # correct
    ("Aisha Rahman", "PER", 23, 24),                    # correct
    ("lightweight vision model", "PROJ", 30, 32),       # out-of-schema FP
]

# Compare SETS of complete entity items.
# Remember: a boundary error or type error usually creates BOTH
# a false positive prediction and a false negative gold entity.

# YOUR CODE HERE:
# Use `set(...)` for predictions and gold items before comparing them.
# For sets A and B, `A - B` means items in A but not in B.
# Keep the three None placeholders until you calculate your answers.
scores_error_analysis_B = None
false_positives_B = None
false_negatives_B = None

# Suggested:
# pred_set = set(error_analysis_predictions_B)
# gold_set = set(gold_entities_B)
# scores_error_analysis_B = prf1(pred_set, gold_set)
# false_positives_B = pred_set - gold_set
# false_negatives_B = gold_set - pred_set

print("Scores:", scores_error_analysis_B)
print("False positives:", false_positives_B)
print("False negatives:", false_negatives_B)

**Your B6 error analysis:**

| Prediction / missing item | Error category | What would you investigate? |
|---|---|---|
| ... | ... | ... |
| ... | ... | ... |
| ... | ... | ... |

Possible things to investigate include annotation rules, tokenisation, span representation, label definitions, candidate generation, negative examples, contextual representation, grounding instructions, or schema design.

---

# Exit ticket

Answer each in **one or two sentences**.

1. **BIO vs span extraction:** When would a span formulation be more natural than BIO?
2. **Entity linking:** What question does linking answer that entity typing does not?
3. **Relation direction:** Why are `(e1, e2)` and `(e2, e1)` different inputs?
4. **Events:** What is the difference between a trigger and an argument?
5. **Structured extraction:** Why should a generated JSON record be validated before storage?
6. **Evaluation:** Why can exact match be harsh but still useful?

### Your exit ticket

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...

## What you should take away

Lecture 5 can be reduced to one structured-prediction story:

```text
text
 ↓
entities
 ↓
linked identities
 ↓
relations / events
 ↓
application records
 ↓
validated structured output
```

The most important distinctions are:

- **Entity extraction = boundary detection + type classification.**
- **BIO and span extraction encode the same basic information differently.**
- NER is a **special case** of broader entity extraction; domain-specific types may also matter.
- **Entity linking** resolves a mention to a canonical identity.
- Relation extraction is classification over an **ordered pair**, so HEAD/TAIL direction matters.
- Pipeline extraction is modular but can **propagate entity errors** into relation extraction.
- Event extraction first identifies **what happened** and then **who/what participated in which role**.
- A schema defines what downstream software expects; `null` is preferable to unsupported guessing.
- Repeated events/projects often require **arrays of records** so relationships are not lost.
- Any extracted JSON record is a **candidate output**, not automatically trustworthy.
- Evaluation must match the output unit: exact span, relation tuple, event structure, record field, or grounded schema-valid JSON.
- Error analysis should identify whether a failure is due to **boundary, type, linking, relation, direction, missing arguments, unsupported values or schema handling**.
